# validate_dw


In [ ]:
"""
Valida que semi2.dm_mortality.* cumpla scripts/dw/ddl_oracle.sql.
Correr DESPUÉS de dimensions.py y facts.py. Pegar en una celda de Databricks.

Chequea, por tabla:
  1. Columnas (nombres) == DDL          (faltantes / extra)
  2. Unicidad de PRIMARY KEY            (count == count distinct PK)
  3. FKs sin NULL                       (los hechos nunca tienen FK nula)
  4. Integridad referencial             (toda FK existe en su dim → anti-join = 0)
  5. Conteo de filas y cobertura de causa
Imprime un reporte con ✓ / ✗ y un veredicto final.
"""

from pyspark.sql import functions as F

SCHEMA = "semi2.dm_mortality"

# ── Contrato (scripts/dw/ddl_oracle.sql) ────────────────────────────────────
DIMS = {
    "dim_causa":       (["id_causa", "cie-10_code", "cie-10_nombre", "gbd_code", "gbd_nombre"], ["id_causa"]),
    "dim_etario":      (["id_etario", "grupo_edad_codigo", "grupo_edad", "categoria_etaria", "edad_minima", "edad_maxima"], ["id_etario"]),
    "dim_genero":      (["id_genero", "sexo_codigo", "sexo_nombre"], ["id_genero"]),
    "dim_geografia":   (["id_geografia", "pais_iso3", "pais_nombre", "dep_ocurrencia", "mun_ocurrencia"], ["id_geografia"]),
    "dim_ihme_perfil": (["id_ihme_perfil", "metrica", "medida"], ["id_ihme_perfil"]),
    "dim_ine_perfil":  (["id_ine_perfil", "estado_civil", "escolaridad", "asistencia_medica", "tipo_ocurrencia"], ["id_ine_perfil"]),
    "dim_source":      (["id_source", "source", "ingestion_ts", "record_hash"], ["id_source"]),
    "dim_tiempo":      (["id_tiempo", "mes_ocurrencia", "mes_ocurrencia_num", "anio_ocurrencia", "es_pre_covid"], ["id_tiempo"]),
}

FACTS = {
    "fact_ihme": (
        ["valor", "limite_inferior", "limite_superior", "id_geografia", "id_genero", "id_causa", "id_tiempo", "id_source", "id_ihme_perfil"],
        ["id_geografia", "id_genero", "id_causa", "id_tiempo", "id_source", "id_ihme_perfil"],
        {"id_geografia": "dim_geografia", "id_genero": "dim_genero", "id_causa": "dim_causa", "id_tiempo": "dim_tiempo", "id_source": "dim_source", "id_ihme_perfil": "dim_ihme_perfil"},
    ),
    "fact_ine": (
        ["defuncion", "id_geografia", "id_tiempo", "id_source", "id_causa", "id_etario", "id_genero", "id_ine_perfil"],
        ["id_geografia", "id_tiempo", "id_source", "id_causa", "id_etario", "id_genero", "id_ine_perfil"],
        {"id_geografia": "dim_geografia", "id_tiempo": "dim_tiempo", "id_source": "dim_source", "id_causa": "dim_causa", "id_etario": "dim_etario", "id_genero": "dim_genero", "id_ine_perfil": "dim_ine_perfil"},
    ),
    "fact_mspas": (
        ["defunciones", "tasa_por_100k", "id_geografia", "id_source", "id_tiempo"],
        ["id_geografia", "id_source", "id_tiempo"],
        {"id_geografia": "dim_geografia", "id_source": "dim_source", "id_tiempo": "dim_tiempo"},
    ),
    "fact_panama": (
        ["id_etario", "id_causa", "id_genero", "id_source", "id_tiempo", "id_geografia", "defunciones"],
        ["id_etario", "id_causa", "id_genero", "id_source", "id_tiempo", "id_geografia"],
        {"id_etario": "dim_etario", "id_causa": "dim_causa", "id_genero": "dim_genero", "id_source": "dim_source", "id_tiempo": "dim_tiempo", "id_geografia": "dim_geografia"},
    ),
    "fact_who_deaths": (
        ["number", "prc_cause_specific_deaths_out_of_total_deaths", "age_std_death_rate_per_100k_std_population", "death_rate_per_100k_population", "id_source", "id_geografia", "id_causa", "id_tiempo", "id_genero"],
        ["id_source", "id_geografia", "id_causa", "id_tiempo", "id_genero"],
        {"id_source": "dim_source", "id_geografia": "dim_geografia", "id_causa": "dim_causa", "id_tiempo": "dim_tiempo", "id_genero": "dim_genero"},
    ),
    "fact_who_population": (
        ["population", "id_source", "id_tiempo", "id_geografia", "id_genero", "id_etario"],
        ["id_source", "id_tiempo", "id_geografia", "id_genero", "id_etario"],
        {"id_source": "dim_source", "id_tiempo": "dim_tiempo", "id_geografia": "dim_geografia", "id_genero": "dim_genero", "id_etario": "dim_etario"},
    ),
}

OK, BAD = "✓", "✗"
_fail = {"n": 0}


def _mark(cond):
    if not cond:
        _fail["n"] += 1
    return OK if cond else BAD


def _col(name):
    return F.col(f"`{name}`")  # backticks: soporta 'cie-10_code'


def _check_columns(df, expected, label):
    actual = df.columns
    missing = [c for c in expected if c not in actual]
    extra = [c for c in actual if c not in expected]
    order_ok = actual == expected
    ok = not missing and not extra
    print(f"  [{_mark(ok)}] columnas: {len(actual)}/{len(expected)}"
          + (f" | FALTAN {missing}" if missing else "")
          + (f" | EXTRA {extra}" if extra else "")
          + ("" if order_ok else " | (orden difiere del DDL)"))


def _check_pk(df, pk, label):
    total = df.count()
    distinct = df.select(*[_col(c) for c in pk]).distinct().count()
    ok = total == distinct
    print(f"  [{_mark(ok)}] PK única: filas={total:,} distinct({','.join(pk)})={distinct:,}"
          + ("" if ok else f"  ← {total - distinct:,} COLISIONES"))
    return total


def _check_fks(df, fks):
    for fk, dim in fks.items():
        nulls = df.filter(_col(fk).isNull()).count()
        dim_ids = spark.table(f"{SCHEMA}.{dim}").select(_col(fk).alias("_id"))
        orphans = (df.select(_col(fk).alias("_id"))
                     .join(dim_ids, "_id", "left_anti").count())
        ok = nulls == 0 and orphans == 0
        print(f"  [{_mark(ok)}] FK {fk} → {dim}: nulls={nulls}  huérfanos={orphans}")


print("=" * 70)
print("VALIDACIÓN semi2.dm_mortality vs scripts/dw/ddl_oracle.sql")
print("=" * 70)

print("\n### DIMENSIONES ###")
for t, (cols, pk) in DIMS.items():
    if not spark.catalog.tableExists(f"{SCHEMA}.{t}"):
        print(f"\n{t}: [{_mark(False)}] NO EXISTE")
        continue
    df = spark.table(f"{SCHEMA}.{t}")
    print(f"\n{t}")
    _check_columns(df, cols, t)
    _check_pk(df, pk, t)

print("\n### HECHOS ###")
for t, (cols, pk, fks) in FACTS.items():
    if not spark.catalog.tableExists(f"{SCHEMA}.{t}"):
        print(f"\n{t}: [{_mark(False)}] NO EXISTE")
        continue
    df = spark.table(f"{SCHEMA}.{t}")
    print(f"\n{t}")
    _check_columns(df, cols, t)
    _check_pk(df, pk, t)
    _check_fks(df, fks)
    if "id_causa" in df.columns:
        tot = df.count()
        na = df.filter(_col("id_causa") == -1).count()
        print(f"  [·] cobertura causa: {tot - na:,}/{tot:,} con causa real ({100*(tot-na)/tot:.1f}%)")

print("\n" + "=" * 70)
if _fail["n"] == 0:
    print("VEREDICTO: ✓ TODO OK — el modelo cumple ddl_oracle.sql")
else:
    print(f"VEREDICTO: ✗ {_fail['n']} chequeo(s) fallaron — revisar arriba")
print("=" * 70)
